In [ ]:
!pip install -q transformers accelerate peft

In [ ]:
import os, random
from glob import glob
from typing import Dict, List, Tuple

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile, UnidentifiedImageError

from transformers import CLIPProcessor, CLIPModel
from peft import LoraConfig, get_peft_model

# ---- PIL safety settings ----
Image.MAX_IMAGE_PIXELS = None          # disable decompression bomb limit
ImageFile.LOAD_TRUNCATED_IMAGES = True # allow truncated images

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
import os
from pathlib import Path
from collections import Counter

# ============================================================
# Dataset root
# ============================================================
DATA_ROOT = "/content/drive/MyDrive/clean_final"

# ============================================================
# Folder -> Label mapping
# ============================================================
FOLDER_TO_LABEL = {
    "UML Activity": "activity",
    "UML Class": "class",
    "UML Communication Diagram": "communication",
    "UML Component Diagram": "component",
    "UML Deployment Diagram": "deployment",
    "UML non-UML": "no_diagram",
    "UML Object Diagram": "object",
    "UML Package Diagram": "package",
    "UML Sequence": "sequence",
    "UML State Machine Diagram": "state_machine",
    "UML Use case": "use_case",
}

# ============================================================
# Inspection
# ============================================================
print("=" * 80)
print("DATASET FILE INSPECTION")
print("=" * 80)

total_all_files = 0
total_all_images = 0
total_code_images = 0

all_extensions = Counter()

# Extensions that we consider images for inspection
IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp",
    ".webp", ".tif", ".tiff", ".gif"
}

for folder_name in FOLDER_TO_LABEL.keys():

    folder_path = Path(DATA_ROOT) / folder_name

    if not folder_path.exists():
        print(f"⚠ Missing folder: {folder_name}")
        continue

    # --------------------------------------------------------
    # ALL files recursively inside this class folder
    # --------------------------------------------------------
    all_files = [
        f for f in folder_path.rglob("*")
        if f.is_file()
    ]

    # --------------------------------------------------------
    # All image-like files, case-insensitive
    # --------------------------------------------------------
    image_files = [
        f for f in all_files
        if f.suffix.lower() in IMAGE_EXTENSIONS
    ]

    # --------------------------------------------------------
    # EXACTLY what your original build_examples() code detects
    # Only direct files and lowercase extensions:
    # png, jpg, jpeg, bmp
    # --------------------------------------------------------
    code_files = []

    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
        code_files.extend(folder_path.glob(ext))

    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------
    total_all_files += len(all_files)
    total_all_images += len(image_files)
    total_code_images += len(code_files)

    for f in all_files:
        all_extensions[f.suffix.lower()] += 1

    difference = len(image_files) - len(code_files)

    print(
        f"{folder_name:32s} | "
        f"Images: {len(image_files):4d} | "
        f"Used by code: {len(code_files):4d} | "
        f"Missing: {difference:3d}"
    )

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(f"All files physically present       : {total_all_files}")
print(f"All recognized image files         : {total_all_images}")
print(f"Images detected by current code    : {total_code_images}")
print(f"Images NOT detected by current code: {total_all_images - total_code_images}")

# ============================================================
# Extensions
# ============================================================
print("\n" + "=" * 80)
print("FILE EXTENSIONS PRESENT")
print("=" * 80)

for ext, count in sorted(all_extensions.items()):
    print(f"{ext or '[no extension]':15s}: {count}")

DATASET FILE INSPECTION
UML Activity                     | Images:  471 | Used by code:  471 | Missing:   0
UML Class                        | Images:  707 | Used by code:  700 | Missing:   7
UML Communication Diagram        | Images:  204 | Used by code:  204 | Missing:   0
UML Component Diagram            | Images:  208 | Used by code:  208 | Missing:   0
UML Deployment Diagram           | Images:  287 | Used by code:  268 | Missing:  19
UML non-UML                      | Images: 4277 | Used by code: 4270 | Missing:   7
UML Object Diagram               | Images:  207 | Used by code:  207 | Missing:   0
UML Package Diagram              | Images:  247 | Used by code:  247 | Missing:   0
UML Sequence                     | Images:  727 | Used by code:  727 | Missing:   0
UML State Machine Diagram        | Images:  323 | Used by code:  323 | Missing:   0
UML Use case                     | Images:  655 | Used by code:  655 | Missing:   0

SUMMARY
All files physically present       : 8314
A

In [ ]:
from pathlib import Path

DATA_ROOT = "/content/drive/MyDrive/clean_final"

FOLDER_TO_LABEL = {
    "UML Activity": "activity",
    "UML Class": "class",
    "UML Communication Diagram": "communication",
    "UML Component Diagram": "component",
    "UML Deployment Diagram": "deployment",
    "UML non-UML": "no_diagram",
    "UML Object Diagram": "object",
    "UML Package Diagram": "package",
    "UML Sequence": "sequence",
    "UML State Machine Diagram": "state_machine",
    "UML Use case": "use_case",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp",
    ".gif", ".webp", ".tif", ".tiff"
}

print("=" * 100)
print("IMAGES NOT DETECTED BY ORIGINAL TRAINING CODE")
print("=" * 100)

total_missing = 0

for folder_name in FOLDER_TO_LABEL:

    folder_path = Path(DATA_ROOT) / folder_name

    # All actual images, recursively and case-insensitive
    all_images = {
        f for f in folder_path.rglob("*")
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    }

    # Exactly what original training code detects
    detected = set()

    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
        detected.update(folder_path.glob(ext))

    missing = sorted(all_images - detected)

    if missing:
        print(f"\n{folder_name} — {len(missing)} missing")
        print("-" * 100)

        for f in missing:

            # Determine why it was missed
            if f.suffix.lower() == ".gif":
                reason = "GIF not included in training extensions"

            elif f.parent != folder_path:
                reason = "Image is inside a subfolder"

            elif f.suffix != f.suffix.lower():
                reason = f"Uppercase/mixed-case extension ({f.suffix})"

            else:
                reason = "Other"

            print(f"{f.name}")
            print(f"   Extension : {f.suffix}")
            print(f"   Reason    : {reason}")
            print(f"   Path      : {f}")

        total_missing += len(missing)

print("\n" + "=" * 100)
print(f"TOTAL IMAGES NOT DETECTED: {total_missing}")
print("=" * 100)

IMAGES NOT DETECTED BY ORIGINAL TRAINING CODE

UML Class — 7 missing
----------------------------------------------------------------------------------------------------
2.JPG
   Extension : .JPG
   Reason    : Uppercase/mixed-case extension (.JPG)
   Path      : /content/drive/MyDrive/clean_final/UML Class/2.JPG
ATMMachine-ClassDiagram.JPG
   Extension : .JPG
   Reason    : Uppercase/mixed-case extension (.JPG)
   Path      : /content/drive/MyDrive/clean_final/UML Class/ATMMachine-ClassDiagram.JPG
ContentManagementSystem-ClassDiagram.JPG
   Extension : .JPG
   Reason    : Uppercase/mixed-case extension (.JPG)
   Path      : /content/drive/MyDrive/clean_final/UML Class/ContentManagementSystem-ClassDiagram.JPG
Purchase-ClassDiagram.JPG
   Extension : .JPG
   Reason    : Uppercase/mixed-case extension (.JPG)
   Path      : /content/drive/MyDrive/clean_final/UML Class/Purchase-ClassDiagram.JPG
classdiagram1(1).JPG
   Extension : .JPG
   Reason    : Uppercase/mixed-case extension (.JPG)
  

In [ ]:
from huggingface_hub import login

login(token="hf_XXXXXXXXXXXXXXXXXXXXXXX")

In [ ]:
import os, random
from typing import Dict, List, Tuple

# ============================================================
# Dataset root
# ============================================================
DATA_ROOT = "/content/drive/MyDrive/clean_final"

# ============================================================
# Folder -> Label mapping
# ============================================================
FOLDER_TO_LABEL = {
    "UML Activity":              "activity",
    "UML Class":                 "class",
    "UML Communication Diagram": "communication",
    "UML Component Diagram":     "component",
    "UML Deployment Diagram":    "deployment",
    "UML Object Diagram":        "object",
    "UML Package Diagram":       "package",
    "UML Sequence":              "sequence",
    "UML State Machine Diagram": "state_machine",
    "UML Use case":              "use_case",
    "UML non-UML":               "non_uml",
}

# ============================================================
# Label mappings
# ============================================================
label_names = sorted(set(FOLDER_TO_LABEL.values()))
label_to_id = {lbl: i for i, lbl in enumerate(label_names)}
id_to_label = {i: lbl for lbl, i in label_to_id.items()}
num_labels = len(label_names)

print("Labels:", label_names)
print("num_labels:", num_labels)


# ============================================================
# Build dataset examples
# ============================================================
def build_examples(
    root_dir: str,
    folder_to_label: Dict[str, str]
) -> List[Tuple[str, int]]:
    """
    Collect image paths and label indices from folders.

    Included:
        .png
        .jpg
        .jpeg
        .bmp

    Uppercase/mixed-case versions such as .JPG are also included.

    Excluded:
        .gif
        .csv
        all other file types
    """

    examples = []

    allowed_extensions = {
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp"
    }

    for folder_name, label_name in folder_to_label.items():

        label_id = label_to_id[label_name]
        folder_path = os.path.join(root_dir, folder_name)

        if not os.path.isdir(folder_path):
            print("⚠️ folder missing:", folder_path)
            continue

        files = []

        # ----------------------------------------------------
        # Read files directly from each class folder
        # ----------------------------------------------------
        for filename in os.listdir(folder_path):

            file_path = os.path.join(folder_path, filename)

            # Only actual files
            if not os.path.isfile(file_path):
                continue

            # Convert extension to lowercase so that
            # .JPG, .JPEG, .PNG, etc. are also detected
            extension = os.path.splitext(filename)[1].lower()

            # Include only approved image extensions
            if extension in allowed_extensions:
                files.append(file_path)

        # ----------------------------------------------------
        # Add image path + corresponding label
        # ----------------------------------------------------
        for f in files:
            examples.append((f, label_id))

    random.shuffle(examples)

    print(f"Total images found (raw): {len(examples)}")

    return examples


# ============================================================
# Train / Validation / Test split
# ============================================================
def split_examples(
    examples: List[Tuple[str, int]],
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
):

    random.seed(seed)

    n = len(examples)

    idx = list(range(n))
    random.shuffle(idx)

    n_train = int(train_ratio * n)
    n_val   = int(val_ratio * n)

    train_idx = idx[:n_train]
    val_idx   = idx[n_train:n_train + n_val]
    test_idx  = idx[n_train + n_val:]

    train_ex = [examples[i] for i in train_idx]
    val_ex   = [examples[i] for i in val_idx]
    test_ex  = [examples[i] for i in test_idx]

    print(
        f"Train: {len(train_ex)}, "
        f"Val: {len(val_ex)}, "
        f"Test: {len(test_ex)}"
    )

    return train_ex, val_ex, test_ex


# ============================================================
# Build + split
# ============================================================
examples = build_examples(
    DATA_ROOT,
    FOLDER_TO_LABEL
)

train_ex, val_ex, test_ex = split_examples(
    examples
)

Labels: ['activity', 'class', 'communication', 'component', 'deployment', 'non_uml', 'object', 'package', 'sequence', 'state_machine', 'use_case']
num_labels: 11
Total images found (raw): 8294
Train: 5805, Val: 1244, Test: 1245


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class UMLImageDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        path, label_id = self.examples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print("🚫 Bad image at runtime, skipping:", path, "|", repr(e))
            new_idx = (idx + 1) % len(self.examples)
            path, label_id = self.examples[new_idx]
            img = Image.open(path).convert("RGB")

        inputs = processor(images=img, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)  # (3, H, W)
        return pixel_values, label_id


train_ds = UMLImageDataset(train_ex)
val_ds   = UMLImageDataset(val_ex)
test_ds  = UMLImageDataset(test_ex)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
import torch
from torch import nn
from transformers import CLIPModel
from peft import LoraConfig, get_peft_model

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

base_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

for p in base_clip.parameters():
    p.requires_grad = False

vision_blocks = base_clip.vision_model.encoder.layers
NUM_UNFREEZE = 12  # last 8 blocks

for blk in vision_blocks[-NUM_UNFREEZE:]:
    for p in blk.parameters():
        p.requires_grad = True

for p in base_clip.visual_projection.parameters():
    p.requires_grad = True

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    task_type="FEATURE_EXTRACTION",
)

base_clip = get_peft_model(base_clip, lora_config)
base_clip.print_trainable_parameters()


class CLIPUMLClassifier(nn.Module):
    def __init__(self, clip_model, num_labels):
        super().__init__()
        self.clip = clip_model
        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, pixel_values):
        img_feats = self.clip.get_image_features(pixel_values=pixel_values)
        logits = self.classifier(img_feats)
        return logits


model = CLIPUMLClassifier(base_clip, num_labels).to(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Device: cuda


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

trainable params: 1,966,080 || all params: 153,243,393 || trainable%: 1.2830


In [ ]:
criterion = nn.CrossEntropyLoss()

trainable_params = list(model.classifier.parameters()) + [
    p for p in model.clip.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(trainable_params, lr=5e-5)

print(
    "Trainable params:",
    sum(p.numel() for p in trainable_params)
)

Trainable params: 1971723


In [ ]:
def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for pixel_values, labels in loader:
        pixel_values = pixel_values.to(device)

        if not isinstance(labels, torch.Tensor):
            labels = torch.tensor(labels, dtype=torch.long, device=device)
        else:
            labels = labels.to(device, dtype=torch.long)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            logits = model(pixel_values)
            loss = criterion(logits, labels)

            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

In [ ]:
import torch
from torch import nn

# Redefine the CLIPUMLClassifier class with the fix for this cell
# Ideally, this change should be made in cell vqORchFEaEnv where the class is originally defined.
class CLIPUMLClassifier(nn.Module):
    def __init__(self, clip_model, num_labels):
        super().__init__()
        self.clip = clip_model
        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, pixel_values):
        img_feats = self.clip.get_image_features(pixel_values=pixel_values)
        # Fix: Access the pooler_output from the BaseModelOutputWithPooling object
        logits = self.classifier(img_feats.pooler_output)
        return logits

# Re-instantiate the model with the corrected class definition
# Ensure 'base_clip', 'num_labels', and 'device' are already defined from previous cells
model = CLIPUMLClassifier(base_clip, num_labels).to(device)

EPOCHS = 10
best_val_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc     = run_epoch(model, val_loader,   train=False)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()

if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc = run_epoch(model, test_loader, train=False)
print(f"Test | loss={test_loss:.4f}, acc={test_acc:.3f}")

SAVE_PATH = "uml_clip_lora_classifier_N12.pt"

torch.save(
    {
        "model_state": model.state_dict(),
        "label_to_id": label_to_id,
        "id_to_label": id_to_label,
    },
    SAVE_PATH,
)

print("✅ Saved fine-tuned model to:", SAVE_PATH)

🚫 Bad image at runtime, skipping: /content/drive/MyDrive/clean_final/UML State Machine Diagram/img_0325 (2).jpg | UnidentifiedImageError("cannot identify image file '/content/drive/MyDrive/clean_final/UML State Machine Diagram/img_0325 (2).jpg'")


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 01 | train_loss=1.3531, train_acc=0.610 | val_loss=0.9272, val_acc=0.774


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


🚫 Bad image at runtime, skipping: /content/drive/MyDrive/clean_final/UML State Machine Diagram/img_0325 (2).jpg | UnidentifiedImageError("cannot identify image file '/content/drive/MyDrive/clean_final/UML State Machine Diagram/img_0325 (2).jpg'")


...

🚫 Bad image at runtime, skipping: /content/drive/MyDrive/clean_final/UML State Machine Diagram/img_0325 (2).jpg | UnidentifiedImageError("cannot identify image file '/content/drive/MyDrive/clean_final/UML State Machine Diagram/img_0325 (2).jpg'")


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 10 | train_loss=0.1055, train_acc=0.994 | val_loss=0.2848, val_acc=0.927
🚫 Bad image at runtime, skipping: /content/drive/MyDrive/clean_final/UML Deployment Diagram/img_0179 (2).jpg | UnidentifiedImageError("cannot identify image file '/content/drive/MyDrive/clean_final/UML Deployment Diagram/img_0179 (2).jpg'")


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96981753 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Test | loss=0.2530, acc=0.939
✅ Saved fine-tuned model to: uml_clip_lora_classifier_N12.pt
